# UK Aviation Demand and Punctuality Analysis

## Notebook 01 — Data Collection and Validation

### Purpose

This notebook establishes the reproducible data foundation for the project. It will collect, inspect and validate monthly airport-passenger and flight-punctuality data published by the UK Civil Aviation Authority.

### Objectives

- Define the project’s data scope
- Identify the required CAA source tables
- Download monthly passenger data
- Inspect file structures and column names
- Standardise and combine the monthly files
- Check missing values, duplicates and data types
- Save a validated interim dataset for later notebooks

### Scope

- **Period:** January 2015 to December 2025
- **Passenger source:** CAA Table 09 — Terminal and Transit Passengers
- **Punctuality source:** CAA Punctuality Statistics Summary Analysis
- **Geographical coverage:** UK reporting airports
- **Forecasting frequency:** Monthly

The incomplete 2026 calendar year is excluded from formal modelling so that the analysis uses complete annual and seasonal cycles.

## 1. Imports and project paths

The notebook uses reusable project-relative paths so that it can run on another computer without exposing or depending on a personal absolute file path.

In [1]:
from pathlib import Path
from urllib.parse import urljoin
import re
import time

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

# Locate the project root whether Jupyter starts from the project
# folder or from the notebooks folder.
current_folder = Path.cwd().resolve()

if (current_folder / "README.md").exists():
    project_root = current_folder
elif (current_folder.parent / "README.md").exists():
    project_root = current_folder.parent
else:
    raise FileNotFoundError("The project root could not be located.")

raw_data_dir = project_root / "data" / "raw"
interim_data_dir = project_root / "data" / "interim"
processed_data_dir = project_root / "data" / "processed"

for directory in [raw_data_dir, interim_data_dir, processed_data_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project: {project_root.name}")
print(f"Raw data folder: {raw_data_dir.relative_to(project_root)}")
print(f"Interim data folder: {interim_data_dir.relative_to(project_root)}")

Project: uk-aviation-demand-punctuality
Raw data folder: data/raw
Interim data folder: data/interim


## 2. Define the collection period and CAA source pages

CAA airport statistics are published on separate monthly web pages. We will first test one completed month before automating collection across 2015–2025.

In [2]:
START_YEAR = 2015
END_YEAR = 2025

MONTH_NAMES = {
    1: "january",
    2: "february",
    3: "march",
    4: "april",
    5: "may",
    6: "june",
    7: "july",
    8: "august",
    9: "september",
    10: "october",
    11: "november",
    12: "december",
}

CAA_AIRPORT_BASE = (
    "https://www.caa.co.uk/data-and-analysis/"
    "uk-aviation-market/airports/uk-airport-data"
)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 UK aviation portfolio analysis "
        "(educational, non-commercial use)"
    )
}

def build_airport_page_url(year, month):
    month_name = MONTH_NAMES[month]
    return (
        f"{CAA_AIRPORT_BASE}/uk-airport-data-{year}/"
        f"{month_name}-{year}/"
    )

sample_page_url = build_airport_page_url(2025, 1)

response = requests.get(
    sample_page_url,
    headers=REQUEST_HEADERS,
    timeout=30,
)
response.raise_for_status()

print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers.get('Content-Type')}")
print(f"Page size: {len(response.content):,} bytes")
print(f"Test page: January 2025")

Status code: 200
Content type: text/html; charset=utf-8
Page size: 39,239 bytes
Test page: January 2025


## 3. Locate the passenger-data download link

The monthly page contains several statistical tables. We need the CSV link for Table 09, which reports terminal and transit passengers by airport.

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

table_09_links = []

for link in soup.find_all("a", href=True):
    link_text = " ".join(link.get_text(" ", strip=True).split())

    if "Table 09" in link_text and "CSV" in link_text:
        table_09_links.append(
            {
                "link_text": link_text,
                "download_url": urljoin(
                    sample_page_url,
                    link["href"],
                ),
            }
        )

table_09_links_df = pd.DataFrame(table_09_links)

print(f"Matching Table 09 CSV links: {len(table_09_links_df)}")
table_09_links_df

Matching Table 09 CSV links: 1


,link_text,download_url
0,"Table 09 Terminal and Transit Passengers (CSV, 6 KB)",https://www.caa.co.uk/Documents/Download/23995/23907051-3e36-4e54-b7aa-53d0296849df/16975


## 4. Download and inspect one sample CSV

Before collecting the full period, one monthly file is downloaded into memory and inspected. This helps identify metadata rows, delimiters, headings and any formatting issues.

In [4]:
sample_download_url = table_09_links_df.loc[0, "download_url"]

csv_response = requests.get(
    sample_download_url,
    headers=REQUEST_HEADERS,
    timeout=30,
)
csv_response.raise_for_status()

sample_csv_text = csv_response.content.decode(
    "utf-8-sig",
    errors="replace",
)

print(f"Status code: {csv_response.status_code}")
print(f"Content type: {csv_response.headers.get('Content-Type')}")
print(f"Downloaded size: {len(csv_response.content):,} bytes")
print("\nFirst 12 lines:\n")

for line_number, line in enumerate(
    sample_csv_text.splitlines()[:12],
    start=1,
):
    print(f"{line_number:>2}: {line}")

Status code: 200
Content type: text/csv
Downloaded size: 6,351 bytes

First 12 lines:

 1: rundate,this_period,last_period,reporting_airport_group_name,reporting_airport_name,total_pax_this_period,total_pax_last_period,total_pax_percent,terminal_pax_this_period,terminal_pax_last_period,terminal_pax_percent,transit_pax_this_period,transit_pax_last_period,transit_pax_percent
 2: 3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,ABERDEEN,149243,147266,1.342469,149243,147266,1.342469,0,0,Null
 3: 3/14/2025 9:47:35 AM,202501,202401,Non UK Reporting Airports,ALDERNEY,2456,2931,-16.206073,2456,2931,-16.206073,0,0,Null
 4: 3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,BARRA,509,477,6.708595,507,473,7.188161,2,4,-50.000000
 5: 3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,BELFAST CITY (GEORGE BEST),165373,153226,7.927506,165373,153212,7.937368,0,14,Null
 6: 3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,BELFAST INTERNATIONAL,390629,364664,7.120253,390629,364664,7.120253,0

## 5. Load the sample into pandas

The CSV has a standard header row and can be loaded directly. The text value `Null` is treated as a missing value.

In [5]:
from io import StringIO

sample_passengers = pd.read_csv(
    StringIO(sample_csv_text),
    na_values=["Null"],
)

print(f"Rows: {sample_passengers.shape[0]:,}")
print(f"Columns: {sample_passengers.shape[1]}")
print("\nColumn names:")
print(sample_passengers.columns.tolist())

sample_passengers.head()

Rows: 50
Columns: 14

Column names:
['rundate', 'this_period', 'last_period', 'reporting_airport_group_name', 'reporting_airport_name', 'total_pax_this_period', 'total_pax_last_period', 'total_pax_percent', 'terminal_pax_this_period', 'terminal_pax_last_period', 'terminal_pax_percent', 'transit_pax_this_period', 'transit_pax_last_period', 'transit_pax_percent']


,rundate,this_period,last_period,reporting_airport_group_name,reporting_airport_name,total_pax_this_period,total_pax_last_period,total_pax_percent,terminal_pax_this_period,terminal_pax_last_period,terminal_pax_percent,transit_pax_this_period,transit_pax_last_period,transit_pax_percent
0,3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,ABERDEEN,149243,147266,1.342469,149243,147266,1.342469,0,0,NaN
1,3/14/2025 9:47:35 AM,202501,202401,Non UK Reporting Airports,ALDERNEY,2456,2931,-16.206073,2456,2931,-16.206073,0,0,NaN
2,3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,BARRA,509,477,6.708595,507,473,7.188161,2,4,-50.0
3,3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,BELFAST CITY (GEORGE BEST),165373,153226,7.927506,165373,153212,7.937368,0,14,NaN
4,3/14/2025 9:47:35 AM,202501,202401,Other UK Airports,BELFAST INTERNATIONAL,390629,364664,7.120253,390629,364664,7.120253,0,0,NaN


## 6. Validate the sample structure

The sample is checked for its reporting period, airport coverage, missing values and duplicate airport-period records. A proper monthly date is also derived from the `YYYYMM` period code.

In [6]:
sample_passengers["period_date"] = pd.to_datetime(
    sample_passengers["this_period"].astype(str),
    format="%Y%m",
)

duplicate_count = sample_passengers.duplicated(
    subset=["this_period", "reporting_airport_name"]
).sum()

print(
    "Reporting period:",
    sample_passengers["period_date"].dt.strftime("%B %Y").unique().tolist(),
)
print(
    "Number of airports:",
    sample_passengers["reporting_airport_name"].nunique(),
)
print("Duplicate airport-period records:", duplicate_count)

print("\nMissing values:")
display(
    sample_passengers.isna()
    .sum()
    .rename("missing")
    .to_frame()
)

print("\nTen airports with the most terminal passengers:")
display(
    sample_passengers[
        [
            "reporting_airport_name",
            "terminal_pax_this_period",
            "terminal_pax_last_period",
            "terminal_pax_percent",
        ]
    ]
    .sort_values(
        "terminal_pax_this_period",
        ascending=False,
    )
    .head(10)
)

Reporting period: ['January 2025']
Number of airports: 50
Duplicate airport-period records: 0

Missing values:


,missing
rundate,0
this_period,0
last_period,0
reporting_airport_group_name,0
reporting_airport_name,0
total_pax_this_period,0
total_pax_last_period,0
total_pax_percent,1
terminal_pax_this_period,0
terminal_pax_last_period,0



Ten airports with the most terminal passengers:


,reporting_airport_name,terminal_pax_this_period,terminal_pax_last_period,terminal_pax_percent
22,HEATHROW,6319244,5996142,5.388498
19,GATWICK,2541908,2370018,7.252688
37,MANCHESTER,1937964,1788581,8.352040
44,STANSTED,1844805,1946934,-5.245632
35,LUTON,1140509,1019360,11.884810
16,EDINBURGH,949376,894538,6.130315
7,BIRMINGHAM,780708,716326,8.987807
10,BRISTOL,560572,519362,7.934735
20,GLASGOW,469823,432447,8.642909
4,BELFAST INTERNATIONAL,390629,364664,7.120253


## 7. Create a reusable Table 09 link finder

The CAA webpage structure may vary over time. This function constructs each monthly page, checks that it loads successfully and returns the single Table 09 CSV link.

In [7]:
def find_table_09_csv_url(year, month):
    page_url = build_airport_page_url(year, month)

    page_response = requests.get(
        page_url,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    page_response.raise_for_status()

    page_soup = BeautifulSoup(
        page_response.text,
        "html.parser",
    )

    matches = []

    for link in page_soup.find_all("a", href=True):
        link_text = " ".join(
            link.get_text(" ", strip=True).split()
        )

        is_table_09 = bool(
            re.search(r"table\s*0?9\b", link_text, flags=re.I)
        )
        is_csv = "csv" in link_text.lower()

        if is_table_09 and is_csv:
            matches.append(
                {
                    "link_text": link_text,
                    "download_url": urljoin(
                        page_url,
                        link["href"],
                    ),
                }
            )

    if len(matches) != 1:
        raise ValueError(
            f"Expected one Table 09 CSV for "
            f"{MONTH_NAMES[month].title()} {year}, "
            f"but found {len(matches)}."
        )

    return matches[0]


oldest_month_test = find_table_09_csv_url(2015, 1)

print("Oldest month tested: January 2015")
print("Matching links found: 1")
print(f"Link text: {oldest_month_test['link_text']}")
print(
    "Official CAA download:",
    oldest_month_test["download_url"].startswith(
        "https://www.caa.co.uk/"
    ),
)

Oldest month tested: January 2015
Matching links found: 1
Link text: Table 09 Terminal and Transit Pax (CSV, 9 KB)
Official CAA download: True


## 8. Download monthly passenger files

Table 09 is downloaded for every month from January 2015 to December 2025. Files are stored locally in `data/raw` and excluded from version control. Existing files are not downloaded again.

In [8]:
def download_table_09(year, month):
    period_code = f"{year}{month:02d}"
    file_name = f"table09_{period_code}.csv"
    file_path = raw_data_dir / file_name

    link_info = find_table_09_csv_url(year, month)

    if file_path.exists():
        status = "already_exists"
    else:
        file_response = requests.get(
            link_info["download_url"],
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        file_response.raise_for_status()

        file_path.write_bytes(file_response.content)
        status = "downloaded"

    return {
        "period": period_code,
        "year": year,
        "month": month,
        "file_name": file_name,
        "status": status,
        "file_size_bytes": file_path.stat().st_size,
        "source_page": build_airport_page_url(year, month),
        "download_url": link_info["download_url"],
    }


download_records = []
download_errors = []

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        try:
            record = download_table_09(year, month)
            download_records.append(record)

        except Exception as error:
            download_errors.append(
                {
                    "year": year,
                    "month": month,
                    "error": str(error),
                }
            )

        # Small pause to avoid sending requests too rapidly.
        time.sleep(0.15)

    print(f"Completed {year}")

download_manifest = pd.DataFrame(download_records)
download_errors_df = pd.DataFrame(download_errors)

print("\nCollection summary")
print(f"Expected files: {(END_YEAR - START_YEAR + 1) * 12}")
print(f"Successful files: {len(download_manifest)}")
print(f"Errors: {len(download_errors_df)}")

download_manifest.head()

Completed 2015
Completed 2016
Completed 2017
Completed 2018
Completed 2019
Completed 2020
Completed 2021
Completed 2022
Completed 2023
Completed 2024
Completed 2025

Collection summary
Expected files: 132
Successful files: 132
Errors: 0


,period,year,month,file_name,status,file_size_bytes,source_page,download_url
0,201501,2015,1,table09_201501.csv,already_exists,9434,https://www.caa.co.uk/data-and-analysis/uk-aviation-market/airports/uk-airport-data/uk-airport-data-2015/january-2015/,https://www.caa.co.uk/Documents/Download/3810/af259f56-0079-4341-9e3d-b76d0556bb58/2779
1,201502,2015,2,table09_201502.csv,already_exists,9607,https://www.caa.co.uk/data-and-analysis/uk-aviation-market/airports/uk-airport-data/uk-airport-data-2015/february-2015/,https://www.caa.co.uk/Documents/Download/3813/02a995b5-e542-48ed-891e-e8472e733d4a/2802
2,201503,2015,3,table09_201503.csv,already_exists,9650,https://www.caa.co.uk/data-and-analysis/uk-aviation-market/airports/uk-airport-data/uk-airport-data-2015/march-2015/,https://www.caa.co.uk/Documents/Download/3815/66912f6c-1e51-4689-907b-9396b22448c1/2825
3,201504,2015,4,table09_201504.csv,already_exists,9615,https://www.caa.co.uk/data-and-analysis/uk-aviation-market/airports/uk-airport-data/uk-airport-data-2015/april-2015/,https://www.caa.co.uk/Documents/Download/3817/a3dfb789-c4da-456d-8c23-24cfa44b7b21/2848
4,201505,2015,5,table09_201505.csv,already_exists,9693,https://www.caa.co.uk/data-and-analysis/uk-aviation-market/airports/uk-airport-data/uk-airport-data-2015/may-2015/,https://www.caa.co.uk/Documents/Download/3819/765d22a1-1ac8-4eb3-9153-d2e056bcd82c/2871


## 9. Audit monthly file schemas

Each downloaded CSV is inspected before combination. This identifies changes in column names, column counts or record counts across publication years.

In [9]:
raw_passenger_files = sorted(
    raw_data_dir.glob("table09_*.csv")
)

schema_ids = {}
schema_records = []

for file_path in raw_passenger_files:
    monthly_data = pd.read_csv(
        file_path,
        na_values=["Null"],
    )

    column_signature = tuple(monthly_data.columns)

    if column_signature not in schema_ids:
        schema_ids[column_signature] = len(schema_ids) + 1

    schema_records.append(
        {
            "file_name": file_path.name,
            "period": file_path.stem.split("_")[-1],
            "rows": monthly_data.shape[0],
            "columns": monthly_data.shape[1],
            "schema_id": schema_ids[column_signature],
        }
    )

schema_audit = pd.DataFrame(schema_records)

print(f"Files audited: {len(schema_audit)}")
print(f"Distinct schemas: {schema_audit['schema_id'].nunique()}")
print(
    f"Rows per file: "
    f"{schema_audit['rows'].min()} to "
    f"{schema_audit['rows'].max()}"
)

display(
    schema_audit.groupby("schema_id")
    .agg(
        file_count=("file_name", "count"),
        first_period=("period", "min"),
        last_period=("period", "max"),
        minimum_rows=("rows", "min"),
        maximum_rows=("rows", "max"),
        column_count=("columns", "first"),
    )
    .reset_index()
)

for columns, schema_id in schema_ids.items():
    print(f"\nSchema {schema_id}:")
    print(list(columns))

Files audited: 132
Distinct schemas: 1
Rows per file: 46 to 57


,schema_id,file_count,first_period,last_period,minimum_rows,maximum_rows,column_count
0,1,132,201501,202512,46,57,14



Schema 1:
['rundate', 'this_period', 'last_period', 'reporting_airport_group_name', 'reporting_airport_name', 'total_pax_this_period', 'total_pax_last_period', 'total_pax_percent', 'terminal_pax_this_period', 'terminal_pax_last_period', 'terminal_pax_percent', 'transit_pax_this_period', 'transit_pax_last_period', 'transit_pax_percent']


## 10. Combine the monthly passenger files

All monthly files use the same schema. They are combined into one longitudinal dataset, with a monthly date and source filename added to support validation and traceability.

In [10]:
period_type_issues = []

for file_path in raw_passenger_files:
    monthly_data = pd.read_csv(
        file_path,
        na_values=["Null"],
    )

    period_numeric = pd.to_numeric(
        monthly_data["this_period"],
        errors="coerce",
    )

    invalid_periods = monthly_data.loc[
        period_numeric.isna()
        | (period_numeric % 1 != 0),
        "this_period",
    ].tolist()

    if (
        monthly_data["this_period"].dtype != "int64"
        or invalid_periods
    ):
        period_type_issues.append(
            {
                "file_name": file_path.name,
                "dtype": str(
                    monthly_data["this_period"].dtype
                ),
                "invalid_values": invalid_periods,
                "unique_values": (
                    monthly_data["this_period"]
                    .dropna()
                    .unique()
                    .tolist()
                ),
            }
        )

period_type_issues_df = pd.DataFrame(period_type_issues)

print(
    "Files with a non-integer period column:",
    len(period_type_issues_df),
)

period_type_issues_df

Files with a non-integer period column: 0


""


In [11]:
invalid_date_codes = []

for file_path in raw_passenger_files:
    monthly_data = pd.read_csv(
        file_path,
        na_values=["Null"],
    )

    for period_value in (
        monthly_data["this_period"]
        .dropna()
        .unique()
    ):
        try:
            pd.to_datetime(
                str(int(period_value)),
                format="%Y%m",
            )
        except ValueError as error:
            invalid_date_codes.append(
                {
                    "file_name": file_path.name,
                    "period_value": period_value,
                    "error": str(error),
                }
            )

invalid_date_codes_df = pd.DataFrame(invalid_date_codes)

print(
    "Invalid YYYYMM period values:",
    len(invalid_date_codes_df),
)

invalid_date_codes_df

Invalid YYYYMM period values: 14


,file_name,period_value,error
0,table09_201605.csv,2016005,"time data ""2016005"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
1,table09_201606.csv,2016006,"time data ""2016006"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
2,table09_201607.csv,2016007,"time data ""2016007"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
3,table09_201608.csv,2016008,"time data ""2016008"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
4,table09_201609.csv,2016009,"time data ""2016009"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
5,table09_201701.csv,2017001,"time data ""2017001"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
6,table09_201702.csv,2017002,"time data ""2017002"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
7,table09_201703.csv,2017003,"time data ""2017003"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
8,table09_201704.csv,2017004,"time data ""2017004"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."
9,table09_201705.csv,2017005,"time data ""2017005"" doesn't match format ""%Y%m"". You might want to try:\n - passing `format` if your strings have..."


In [12]:
monthly_frames = []
period_corrections = []

for file_path in raw_passenger_files:
    monthly_data = pd.read_csv(
        file_path,
        na_values=["Null"],
    )

    # The filename was created from the year and month
    # requested from the official CAA monthly webpage.
    expected_period = int(
        file_path.stem.split("_")[-1]
    )
    expected_year = expected_period // 100
    expected_month = expected_period % 100
    expected_last_period = (
        (expected_year - 1) * 100
        + expected_month
    )

    # Preserve the values supplied in the CAA files.
    monthly_data["reported_this_period"] = (
        monthly_data["this_period"]
    )
    monthly_data["reported_last_period"] = (
        monthly_data["last_period"]
    )

    incorrect_period_mask = (
        monthly_data["this_period"]
        != expected_period
    )

    if incorrect_period_mask.any():
        period_corrections.append(
            {
                "file_name": file_path.name,
                "reported_period": (
                    monthly_data.loc[
                        incorrect_period_mask,
                        "this_period",
                    ]
                    .unique()
                    .tolist()
                ),
                "corrected_period": expected_period,
            }
        )

    # Use consistent six-digit YYYYMM values.
    monthly_data["this_period"] = expected_period
    monthly_data["last_period"] = expected_last_period

    monthly_data["period_date"] = pd.to_datetime(
        str(expected_period),
        format="%Y%m",
    )
    monthly_data["source_file"] = file_path.name

    monthly_frames.append(monthly_data)

passengers = (
    pd.concat(monthly_frames, ignore_index=True)
    .sort_values(
        ["period_date", "reporting_airport_name"]
    )
    .reset_index(drop=True)
)

period_corrections_df = pd.DataFrame(
    period_corrections
)

duplicate_count = passengers.duplicated(
    subset=[
        "period_date",
        "reporting_airport_name",
    ]
).sum()

print(f"Combined rows: {passengers.shape[0]:,}")
print(f"Combined columns: {passengers.shape[1]}")
print(
    "Date range:",
    passengers["period_date"]
    .min()
    .strftime("%B %Y"),
    "to",
    passengers["period_date"]
    .max()
    .strftime("%B %Y"),
)
print(
    "Unique reporting airports:",
    passengers[
        "reporting_airport_name"
    ].nunique(),
)
print(
    "Duplicate airport-month records:",
    duplicate_count,
)
print(
    "Files requiring period correction:",
    len(period_corrections_df),
)

passengers.head()

Combined rows: 6,758
Combined columns: 18
Date range: January 2015 to December 2025
Unique reporting airports: 114
Duplicate airport-month records: 0
Files requiring period correction: 14


,rundate,this_period,last_period,reporting_airport_group_name,reporting_airport_name,total_pax_this_period,total_pax_last_period,total_pax_percent,terminal_pax_this_period,terminal_pax_last_period,terminal_pax_percent,transit_pax_this_period,transit_pax_last_period,transit_pax_percent,reported_this_period,reported_last_period,period_date,source_file
0,13-Mar-2015 14:19,201501,201401,Other UK Airports,ABERDEEN,249165,247286,0.759849,249165,247255,0.772482,0,31,NaN,201501,201401,2015-01-01,table09_201501.csv
1,13-Mar-2015 14:19,201501,201401,Non UK Reporting Airports,ALDERNEY,2906,3358,-13.46039,2906,3358,-13.46039,0,0,NaN,201501,201401,2015-01-01,table09_201501.csv
2,13-Mar-2015 14:19,201501,201401,Other UK Airports,BARRA,437,520,-15.96154,437,520,-15.96154,0,0,NaN,201501,201401,2015-01-01,table09_201501.csv
3,13-Mar-2015 14:19,201501,201401,Other UK Airports,BELFAST CITY (GEORGE BEST),167662,162358,3.266855,167662,162358,3.266855,0,0,NaN,201501,201401,2015-01-01,table09_201501.csv
4,13-Mar-2015 14:19,201501,201401,Other UK Airports,BELFAST INTERNATIONAL,237948,251009,-5.203399,237819,251008,-5.254414,129,1,12800.0,201501,201401,2015-01-01,table09_201501.csv


## 11. Audit coverage, airport groups and missing values

The combined dataset is checked for missing calendar months, reporting-group coverage and missing values. This determines which records belong within the UK-airport scope and which fields require cleaning.

In [13]:
expected_months = pd.date_range(
    start=f"{START_YEAR}-01-01",
    end=f"{END_YEAR}-12-01",
    freq="MS",
)

observed_months = pd.DatetimeIndex(
    passengers["period_date"].unique()
).sort_values()

missing_months = expected_months.difference(
    observed_months
)

print(f"Expected months: {len(expected_months)}")
print(f"Observed months: {len(observed_months)}")
print(f"Missing months: {len(missing_months)}")

print("\nAirport-group summary:")
airport_group_summary = (
    passengers.groupby(
        "reporting_airport_group_name"
    )
    .agg(
        records=("reporting_airport_name", "size"),
        unique_airports=(
            "reporting_airport_name",
            "nunique",
        ),
        first_month=("period_date", "min"),
        last_month=("period_date", "max"),
        terminal_passengers=(
            "terminal_pax_this_period",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "terminal_passengers",
        ascending=False,
    )
)

display(airport_group_summary)

print("\nMissing values:")
missing_summary = (
    passengers.isna()
    .sum()
    .rename("missing")
    .to_frame()
)

display(
    missing_summary[
        missing_summary["missing"] > 0
    ]
)

Expected months: 132
Observed months: 132
Missing months: 0

Airport-group summary:


,reporting_airport_group_name,records,unique_airports,first_month,last_month,terminal_passengers
0,London Area Airports,696,6,2016-04-01,2025-12-01,1401674498
4,Other UK Airports,4754,47,2016-04-01,2025-12-01,908917142
1,London Area Airports,90,6,2015-01-01,2016-03-01,189480363
5,Other UK Airports,690,47,2015-01-01,2016-03-01,115615461
2,Non UK Reporting Airports,468,4,2016-04-01,2025-12-01,25763040
3,Non UK Reporting Airports,60,4,2015-01-01,2016-03-01,3896035



Missing values:


,missing
total_pax_percent,298
terminal_pax_percent,301
transit_pax_percent,4675


In [14]:
print("Original airport-group values:")

for group_name in passengers[
    "reporting_airport_group_name"
].unique():
    print(repr(group_name))

passengers["reporting_airport_group_name"] = (
    passengers["reporting_airport_group_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("\nCleaned airport-group values:")

for group_name in passengers[
    "reporting_airport_group_name"
].unique():
    print(repr(group_name))

cleaned_group_summary = (
    passengers.groupby("reporting_airport_group_name")
    .agg(
        records=("reporting_airport_name", "size"),
        unique_airports=(
            "reporting_airport_name",
            "nunique",
        ),
        first_month=("period_date", "min"),
        last_month=("period_date", "max"),
        terminal_passengers=(
            "terminal_pax_this_period",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "terminal_passengers",
        ascending=False,
    )
)

display(cleaned_group_summary)

Original airport-group values:
'Other UK Airports             '
'Non UK Reporting Airports     '
'London Area Airports          '
'Other UK Airports'
'Non UK Reporting Airports'
'London Area Airports'

Cleaned airport-group values:
'Other UK Airports'
'Non UK Reporting Airports'
'London Area Airports'


,reporting_airport_group_name,records,unique_airports,first_month,last_month,terminal_passengers
0,London Area Airports,786,12,2015-01-01,2025-12-01,1591154861
2,Other UK Airports,5444,94,2015-01-01,2025-12-01,1024532603
1,Non UK Reporting Airports,528,8,2015-01-01,2025-12-01,29659075


In [15]:
# Clean airport names
passengers["reporting_airport_name"] = (
    passengers["reporting_airport_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Keep UK reporting airports only
uk_passengers = passengers[
    passengers["reporting_airport_group_name"]
    != "Non UK Reporting Airports"
].copy()

print(
    f"All records before filtering: "
    f"{len(passengers):,}"
)

print(
    f"UK records after filtering: "
    f"{len(uk_passengers):,}"
)

print(
    f"Unique UK airports: "
    f"{uk_passengers['reporting_airport_name'].nunique()}"
)

uk_group_summary = (
    uk_passengers.groupby(
        "reporting_airport_group_name"
    )
    .agg(
        records=("reporting_airport_name", "size"),
        unique_airports=(
            "reporting_airport_name",
            "nunique",
        ),
        first_month=("period_date", "min"),
        last_month=("period_date", "max"),
        terminal_passengers=(
            "terminal_pax_this_period",
            "sum",
        ),
    )
    .reset_index()
)

display(uk_group_summary)

All records before filtering: 6,758
UK records after filtering: 6,230
Unique UK airports: 55


,reporting_airport_group_name,records,unique_airports,first_month,last_month,terminal_passengers
0,London Area Airports,786,6,2015-01-01,2025-12-01,1591154861
1,Other UK Airports,5444,49,2015-01-01,2025-12-01,1024532603


## 12. Audit airport reporting coverage

Not every airport necessarily reported passenger numbers in every month. This section measures the number and percentage of months available for each airport.

In [16]:
total_months = len(expected_months)

airport_coverage = (
    uk_passengers.groupby(
        [
            "reporting_airport_group_name",
            "reporting_airport_name",
        ]
    )
    .agg(
        months_reported=("period_date", "nunique"),
        first_month=("period_date", "min"),
        last_month=("period_date", "max"),
        total_terminal_passengers=(
            "terminal_pax_this_period",
            "sum",
        ),
    )
    .reset_index()
)

airport_coverage["coverage_percentage"] = (
    airport_coverage["months_reported"]
    / total_months
    * 100
).round(1)

airport_coverage = airport_coverage.sort_values(
    [
        "months_reported",
        "total_terminal_passengers",
    ],
    ascending=[False, False],
)

print(
    "Airports reporting all 132 months: "
    f"{(airport_coverage['months_reported'] == total_months).sum()}"
)

print(
    "Airports with incomplete coverage: "
    f"{(airport_coverage['months_reported'] < total_months).sum()}"
)

display(airport_coverage)

Airports reporting all 132 months: 39
Airports with incomplete coverage: 16


,reporting_airport_group_name,reporting_airport_name,months_reported,first_month,last_month,total_terminal_passengers,coverage_percentage
1,London Area Airports,HEATHROW,132,2015-01-01,2025-12-01,739832158,100.0
0,London Area Airports,GATWICK,132,2015-01-01,2025-12-01,397753494,100.0
40,Other UK Airports,MANCHESTER,132,2015-01-01,2025-12-01,259516322,100.0
5,London Area Airports,STANSTED,132,2015-01-01,2025-12-01,254211501,100.0
3,London Area Airports,LUTON,132,2015-01-01,2025-12-01,152527578,100.0
25,Other UK Airports,EDINBURGH,132,2015-01-01,2025-12-01,130786170,100.0
12,Other UK Airports,BIRMINGHAM,132,2015-01-01,2025-12-01,112827598,100.0
15,Other UK Airports,BRISTOL,132,2015-01-01,2025-12-01,83857576,100.0
28,Other UK Airports,GLASGOW,132,2015-01-01,2025-12-01,80582845,100.0
9,Other UK Airports,BELFAST INTERNATIONAL,132,2015-01-01,2025-12-01,56181447,100.0


In [17]:
incomplete_airports = (
    airport_coverage[
        airport_coverage["months_reported"]
        < total_months
    ]
    .sort_values(
        [
            "months_reported",
            "total_terminal_passengers",
        ],
        ascending=[False, False],
    )
)

display(incomplete_airports)

,reporting_airport_group_name,reporting_airport_name,months_reported,first_month,last_month,total_terminal_passengers,coverage_percentage
11,Other UK Airports,BIGGIN HILL,130,2015-01-01,2025-12-01,38151,98.5
4,London Area Airports,SOUTHEND,126,2015-01-01,2025-12-01,8120707,95.5
13,Other UK Airports,BLACKPOOL,126,2015-01-01,2025-12-01,200232,95.5
54,Other UK Airports,WICK JOHN O GROATS,122,2015-01-01,2025-12-01,127711,92.4
21,Other UK Airports,DONCASTER SHEFFIELD,107,2015-01-01,2023-11-01,7685164,81.1
47,Other UK Airports,SCATSTA,78,2015-01-01,2021-06-01,906511,59.1
39,Other UK Airports,LYDD,74,2015-01-01,2025-12-01,2879,56.1
52,Other UK Airports,TEESSIDE INTERNATIONAL AIRPORT,73,2019-12-01,2025-12-01,1007980,55.3
23,Other UK Airports,DURHAM TEES VALLEY,59,2015-01-01,2019-11-01,676466,44.7
45,Other UK Airports,OXFORD (KIDLINGTON),48,2015-02-01,2025-09-01,1933,36.4


### Standardise renamed airports

Durham Tees Valley Airport was renamed Teesside International Airport. The two names form one continuous monthly series: the former name ends in November 2019 and the latter begins in December 2019.

In [18]:
airport_name_replacements = {
    "DURHAM TEES VALLEY":
        "TEESSIDE INTERNATIONAL AIRPORT",
}

uk_passengers["reporting_airport_name"] = (
    uk_passengers["reporting_airport_name"]
    .replace(airport_name_replacements)
)

duplicate_count = uk_passengers.duplicated(
    subset=[
        "reporting_airport_name",
        "period_date",
    ]
).sum()

print(
    "Duplicate airport-month records after "
    f"name standardisation: {duplicate_count}"
)

airport_coverage = (
    uk_passengers.groupby(
        [
            "reporting_airport_group_name",
            "reporting_airport_name",
        ]
    )
    .agg(
        months_reported=("period_date", "nunique"),
        first_month=("period_date", "min"),
        last_month=("period_date", "max"),
        total_terminal_passengers=(
            "terminal_pax_this_period",
            "sum",
        ),
    )
    .reset_index()
)

airport_coverage["coverage_percentage"] = (
    airport_coverage["months_reported"]
    / total_months
    * 100
).round(1)

print(
    "Unique UK airports after standardisation: "
    f"{airport_coverage['reporting_airport_name'].nunique()}"
)

print(
    "Airports reporting all 132 months: "
    f"{(airport_coverage['months_reported'] == total_months).sum()}"
)

print(
    "Airports with incomplete coverage: "
    f"{(airport_coverage['months_reported'] < total_months).sum()}"
)

display(
    airport_coverage[
        airport_coverage["reporting_airport_name"]
        == "TEESSIDE INTERNATIONAL AIRPORT"
    ]
)

Duplicate airport-month records after name standardisation: 0
Unique UK airports after standardisation: 54
Airports reporting all 132 months: 40
Airports with incomplete coverage: 14


,reporting_airport_group_name,reporting_airport_name,months_reported,first_month,last_month,total_terminal_passengers,coverage_percentage
51,Other UK Airports,TEESSIDE INTERNATIONAL AIRPORT,132,2015-01-01,2025-12-01,1684446,100.0


## 13. Create and export the cleaned datasets

The cleaned passenger dataset retains UK reporting airports, standardised airport names and the principal passenger measures. Airport coverage is exported separately so later analysis can distinguish complete and incomplete histories.

In [19]:
complete_airports = set(
    airport_coverage.loc[
        airport_coverage["months_reported"]
        == total_months,
        "reporting_airport_name",
    ]
)

clean_passengers = (
    uk_passengers[
        [
            "period_date",
            "reporting_airport_group_name",
            "reporting_airport_name",
            "total_pax_this_period",
            "terminal_pax_this_period",
            "transit_pax_this_period",
            "source_file",
        ]
    ]
    .rename(
        columns={
            "reporting_airport_group_name":
                "airport_group",
            "reporting_airport_name":
                "airport_name",
            "total_pax_this_period":
                "total_passengers",
            "terminal_pax_this_period":
                "terminal_passengers",
            "transit_pax_this_period":
                "transit_passengers",
        }
    )
    .sort_values(
        ["airport_name", "period_date"]
    )
    .reset_index(drop=True)
)

clean_passengers["complete_history"] = (
    clean_passengers["airport_name"]
    .isin(complete_airports)
)

coverage_export = airport_coverage.rename(
    columns={
        "reporting_airport_group_name":
            "airport_group",
        "reporting_airport_name":
            "airport_name",
    }
).sort_values(
    [
        "months_reported",
        "total_terminal_passengers",
    ],
    ascending=[False, False],
)

interim_dir = project_root / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

passenger_output_path = (
    interim_dir
    / "caa_monthly_airport_passengers_2015_2025.csv"
)

coverage_output_path = (
    interim_dir
    / "airport_reporting_coverage.csv"
)

clean_passengers.to_csv(
    passenger_output_path,
    index=False,
)

coverage_export.to_csv(
    coverage_output_path,
    index=False,
)

print(f"Clean passenger rows: {len(clean_passengers):,}")
print(
    "Unique airports: "
    f"{clean_passengers['airport_name'].nunique()}"
)
print(
    "Complete-history airports: "
    f"{clean_passengers.loc[
        clean_passengers['complete_history'],
        'airport_name'
    ].nunique()}"
)
print(
    "Duplicate airport-month records: "
    f"{clean_passengers.duplicated(
        subset=['airport_name', 'period_date']
    ).sum()}"
)
print(
    "Missing terminal-passenger values: "
    f"{clean_passengers[
        'terminal_passengers'
    ].isna().sum()}"
)

print("\nExported files:")
print(passenger_output_path.relative_to(project_root))
print(coverage_output_path.relative_to(project_root))

display(clean_passengers.head())

Clean passenger rows: 6,230
Unique airports: 54
Complete-history airports: 40
Duplicate airport-month records: 0
Missing terminal-passenger values: 0

Exported files:
data/interim/caa_monthly_airport_passengers_2015_2025.csv
data/interim/airport_reporting_coverage.csv


,period_date,airport_group,airport_name,total_passengers,terminal_passengers,transit_passengers,source_file,complete_history
0,2015-01-01,Other UK Airports,ABERDEEN,249165,249165,0,table09_201501.csv,True
1,2015-02-01,Other UK Airports,ABERDEEN,250294,250294,0,table09_201502.csv,True
2,2015-03-01,Other UK Airports,ABERDEEN,281553,281553,0,table09_201503.csv,True
3,2015-04-01,Other UK Airports,ABERDEEN,298203,298196,7,table09_201504.csv,True
4,2015-05-01,Other UK Airports,ABERDEEN,297671,297501,170,table09_201505.csv,True


## 14. Validate the exported dataset

The exported passenger file is reloaded from disk to confirm that it can be used independently by subsequent notebooks without losing records or introducing data-quality problems.

In [20]:
reloaded_passengers = pd.read_csv(
    passenger_output_path,
    parse_dates=["period_date"],
)

final_validation = {
    "rows": len(reloaded_passengers),
    "columns": reloaded_passengers.shape[1],
    "unique_airports":
        reloaded_passengers["airport_name"].nunique(),
    "unique_months":
        reloaded_passengers["period_date"].nunique(),
    "first_month":
        reloaded_passengers["period_date"].min(),
    "last_month":
        reloaded_passengers["period_date"].max(),
    "duplicate_airport_months":
        reloaded_passengers.duplicated(
            subset=["airport_name", "period_date"]
        ).sum(),
    "missing_terminal_passengers":
        reloaded_passengers[
            "terminal_passengers"
        ].isna().sum(),
}

for check, result in final_validation.items():
    print(f"{check}: {result}")

assert len(reloaded_passengers) == len(clean_passengers)
assert (
    reloaded_passengers["airport_name"].nunique()
    == 54
)
assert (
    reloaded_passengers["period_date"].nunique()
    == 132
)
assert (
    final_validation["duplicate_airport_months"]
    == 0
)
assert (
    final_validation["missing_terminal_passengers"]
    == 0
)

print("\nAll final validation checks passed.")

rows: 6230
columns: 8
unique_airports: 54
unique_months: 132
first_month: 2015-01-01 00:00:00
last_month: 2025-12-01 00:00:00
duplicate_airport_months: 0
missing_terminal_passengers: 0

All final validation checks passed.


## 15. Conclusions

The UK CAA Table 09 passenger data was successfully collected for every month from January 2015 to December 2025.

### Final dataset

- 132 monthly reporting periods
- 6,230 UK airport-month records
- 54 standardised UK airports
- 40 airports with complete 132-month histories
- No duplicate airport-month records
- No missing terminal-passenger values

### Data-quality findings

- Fourteen source files contained malformed seven-digit reporting-period values. The correct periods were derived from the verified source filenames while the original values were retained during processing for audit purposes.
- Older source files contained trailing spaces in airport and airport-group names. These were removed.
- Non-UK reporting airports were excluded from the cleaned dataset.
- Durham Tees Valley and Teesside International Airport were standardised as one continuous airport history.
- Some smaller airports have incomplete reporting histories. Their coverage is documented separately and they have not been treated as missing observations.
- Missing percentage-change fields in the source data were not retained because they can be recalculated from the passenger counts where appropriate.

### Exported datasets

- `data/interim/caa_monthly_airport_passengers_2015_2025.csv`
- `data/interim/airport_reporting_coverage.csv`

These generated files are excluded from version control. The next notebook will use them to explore airport rankings, long-term passenger trends, seasonality and the effects of the COVID-19 disruption.